# Fine-tune Qwen2-VL on LaTeX OCR (Unsloth, Colab)

Image of a formula in, LaTeX out. Runtime → Change runtime type → T4 GPU (the 2B model) or an L4/A100 (the 7B model).

In [ ]:
!pip install -q unsloth

In [ ]:
from datasets import load_dataset
from trl import SFTConfig, SFTTrainer
from unsloth import FastVisionModel, is_bf16_supported
from unsloth.trainer import UnslothVisionDataCollator

MODEL = "unsloth/Qwen2-VL-2B-Instruct"
INSTRUCTION = "Write the LaTeX representation for this image."

model, tokenizer = FastVisionModel.from_pretrained(MODEL, load_in_4bit=True, use_gradient_checkpointing="unsloth")
model = FastVisionModel.get_peft_model(model, finetune_vision_layers=True, finetune_language_layers=True,
    finetune_attention_modules=True, finetune_mlp_modules=True,
    r=16, lora_alpha=16, lora_dropout=0, bias="none", random_state=3407)

In [ ]:
def to_conversation(sample):
    return {"messages": [
        {"role": "user", "content": [{"type": "text", "text": INSTRUCTION}, {"type": "image", "image": sample["image"]}]},
        {"role": "assistant", "content": [{"type": "text", "text": sample["text"]}]},
    ]}

train = load_dataset("unsloth/LaTeX_OCR", split="train")
test = load_dataset("unsloth/LaTeX_OCR", split="test")
train_conv = [to_conversation(s) for s in train]

In [ ]:
def transcribe(image):
    msgs = [{"role": "user", "content": [{"type": "image"}, {"type": "text", "text": INSTRUCTION}]}]
    text = tokenizer.apply_chat_template(msgs, add_generation_prompt=True)
    inputs = tokenizer(image, text, add_special_tokens=False, return_tensors="pt").to("cuda")
    out = model.generate(**inputs, max_new_tokens=256, do_sample=False)
    return tokenizer.decode(out[0, inputs["input_ids"].shape[1]:], skip_special_tokens=True).strip()

FastVisionModel.for_inference(model)
print("before:", transcribe(test[0]["image"]), "\nref:   ", test[0]["text"])

In [ ]:
FastVisionModel.for_training(model)
trainer = SFTTrainer(model=model, tokenizer=tokenizer, data_collator=UnslothVisionDataCollator(model, tokenizer),
    train_dataset=train_conv,
    args=SFTConfig(per_device_train_batch_size=2, gradient_accumulation_steps=4, warmup_steps=5, max_steps=60,
        learning_rate=2e-4, fp16=not is_bf16_supported(), bf16=is_bf16_supported(), logging_steps=5,
        optim="adamw_8bit", weight_decay=0.01, lr_scheduler_type="linear", seed=3407,
        output_dir="outputs/qwen2-vl-latex", report_to="none", remove_unused_columns=False,
        dataset_text_field="", dataset_kwargs={"skip_prepare_dataset": True}, max_seq_length=2048))
trainer.train()

In [ ]:
FastVisionModel.for_inference(model)
print("after: ", transcribe(test[0]["image"]), "\nref:   ", test[0]["text"])
model.save_pretrained("outputs/qwen2-vl-latex/adapter")
tokenizer.save_pretrained("outputs/qwen2-vl-latex/adapter")

Score the adapter with the repo's eval (it loads the base model with transformers and applies the adapter):

```bash
PYTHONPATH=src python eval/run_eval.py --model Qwen/Qwen2-VL-2B-Instruct --adapter outputs/qwen2-vl-latex/adapter --limit 200
```